# Task 2 N-Gram Language Model Demo on Colab

This notebook is intentionally kept small.

Its goal is only to support **Task 2: build a basic prediction model** with:

- unigram / bigram / trigram
- Laplace smoothing
- next-token prediction
- sentence scoring
- perplexity

In [ ]:
from pathlib import Path
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    %cd /content
    if not Path("/content/text-preprocess-tokenization").exists():
        !git clone https://github.com/HatakekkSheeshh/text-preprocess-tokenization.git
    %cd /content/text-preprocess-tokenization
else:
    print("This notebook is not running in Colab. Make sure your working directory is the project root.")

PROJECT_ROOT = Path.cwd()
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import json
import pandas as pd

from src.datasets.load_data import load
from src.training.train_ngram import NGramTrainingConfig, train_ngram_language_model

METRICS_ROOT = PROJECT_ROOT / "outputs" / "metrics" / "ngram"


def load_metrics(run_name: str) -> dict:
    metrics_path = METRICS_ROOT / f"{run_name}.json"
    return json.loads(metrics_path.read_text(encoding="utf-8"))

In [ ]:
# Keep this notebook focused on one simple model run.
DATASET_NAME = "text8"
TOKENIZER_NAME = "word"
NGRAM_ORDER = 3
LAPLACE_ALPHA = 1.0

# Use subset limits for quick Colab experiments.
MAX_FIT_TEXTS = None
MAX_TRAIN_TOKENS = 4096
MAX_VALIDATION_TOKENS = 1024
MAX_TEST_TOKENS = 1024

PREDICTION_CONTEXTS = ["the history of"]
SCORE_TEXTS = [
    "the history of science",
    "science of history the",
]

load(DATASET_NAME)
print(f"{DATASET_NAME} is ready.")

run_name = f"colab_task2_{DATASET_NAME.replace('-', '_')}_{TOKENIZER_NAME}_{NGRAM_ORDER}gram"
config = NGramTrainingConfig(
    dataset_name=DATASET_NAME,
    tokenizer_name=TOKENIZER_NAME,
    order=NGRAM_ORDER,
    alpha=LAPLACE_ALPHA,
    max_vocab_size=None if TOKENIZER_NAME == "char" else 50000,
    max_fit_texts=MAX_FIT_TEXTS,
    max_train_tokens=MAX_TRAIN_TOKENS,
    max_validation_tokens=MAX_VALIDATION_TOKENS,
    max_test_tokens=MAX_TEST_TOKENS,
    run_name=run_name,
)
config

In [ ]:
summary = train_ngram_language_model(
    config,
    prediction_contexts=PREDICTION_CONTEXTS,
    score_texts=SCORE_TEXTS,
    top_k=5,
)
summary["splits"]

In [ ]:
metrics = load_metrics(summary["run_name"])

overview = {
    "dataset": metrics["config"]["dataset_name"],
    "tokenizer": metrics["tokenizer"]["type"],
    "order": metrics["model"]["order"],
    "train_ppl": round(metrics["splits"]["train"]["perplexity"], 4),
    "val_ppl": round(metrics["splits"]["validation"]["perplexity"], 4),
    "test_ppl": round(metrics["splits"]["test"]["perplexity"], 4),
}
display(pd.DataFrame([overview]))

print("Prediction examples:")
for item in metrics["prediction_contexts"]:
    print(item["context_text"])
    for pred in item["predictions"]:
        print("  ", pred["token"], pred["probability"])

print("\nSentence scoring examples:")
for item in metrics["scored_texts"]:
    print(item["text"], "-> ppl =", round(item["perplexity"], 4))